In [19]:
import pandas as pd
import numpy as np

# -----------------------------
# Load towns/cities table
# -----------------------------
towns_path = "/content/trinidad_and_tobago_all_cities_towns.csv"
towns = pd.read_csv(towns_path)

num_customers = 25000  # number of synthetic customers

# -----------------------------
# Customer demographic options
# -----------------------------
marital_statuses = ["Single", "Married", "Divorced", "Widowed"]
marital_weights = [0.45, 0.4, 0.1, 0.05]

professions = ["Working", "Retired", "Student", "Business Owner", "Homemaker", "Unemployed"]

education_levels = ["High School", "Bachelor", "Master", "PhD"]
education_weights = [0.4, 0.35, 0.2, 0.05]

religions = ["Christian", "Hindu", "Muslim", "Other", "None"]
religion_weights = [0.3, 0.3, 0.3, 0.05, 0.05]

# -----------------------------
# Helper functions
# -----------------------------
def random_date_joined(n):
    """Generate n random dates between 2016-01-01 and 2025-01-01"""
    start = pd.to_datetime('2016-01-01')
    end = pd.to_datetime('2025-01-01')
    delta_days = (end - start).days
    random_days = np.random.randint(0, delta_days, n)
    return start + pd.to_timedelta(random_days, unit='d')

# -----------------------------
# Generate base demographic arrays
# -----------------------------
customer_ids = np.arange(1, num_customers + 1)
city_ids = np.random.choice(towns['city_id'], num_customers)
ages = np.random.randint(18, 90, num_customers)
genders = np.random.choice(["Male", "Female"], num_customers)
marital_status = np.random.choice(marital_statuses, num_customers, p=marital_weights)
religions_arr = np.random.choice(religions, num_customers, p=religion_weights)
date_joined = random_date_joined(num_customers)

# -----------------------------
# Assign professions based on age
# -----------------------------
professions_arr = np.empty(num_customers, dtype=object)
for i, age in enumerate(ages):
    if age >= 65:
        professions_arr[i] = "Retired"
    elif age <= 25:
        professions_arr[i] = np.random.choice(["Student", "Working"], p=[0.7, 0.3])
    else:
        adult_probs = [0.5, 0.0, 0.0, 0.15, 0.25, 0.1]
        professions_arr[i] = np.random.choice(professions, p=adult_probs)

# -----------------------------
# Assign education based on profession
# -----------------------------
education_arr = np.empty(num_customers, dtype=object)
for i, prof in enumerate(professions_arr):
    if prof == "Student":
        education_arr[i] = np.random.choice(["Bachelor", "Master"], p=[0.7, 0.3])
    else:
        education_arr[i] = np.random.choice(education_levels, p=education_weights)

# -----------------------------
# Assign income based on profession
# -----------------------------
minimum_wage = 20.50
hours_per_month = 160
minimum_salary = minimum_wage * hours_per_month

income_arr = np.empty(num_customers, dtype=int)
for i, prof in enumerate(professions_arr):
    if prof == "Retired":
        income = int(np.random.normal(25000, 10000))
    elif prof == "Student":
        income = int(np.random.normal(12000, 5000))
        income = max(income, minimum_salary)
    elif prof == "Unemployed":
        income = 0
    elif prof == "Business Owner":
        income = int(np.random.normal(120000, 30000))
        income = max(income, minimum_salary)
    else:
        income = int(np.random.normal(60000, 15000))
        income = max(income, minimum_salary)
    income_arr[i] = income

# -----------------------------
# Assign household size based on age and profession
# -----------------------------
household_size_arr = np.empty(num_customers, dtype=int)
for i, (age, prof) in enumerate(zip(ages, professions_arr)):
    if prof == "Student":
        household_size_arr[i] = np.random.choice([2,3,4,5,6], p=[0.1,0.2,0.3,0.3,0.1])
    elif age <= 25:
        household_size_arr[i] = np.random.choice([1,2,3], p=[0.5,0.3,0.2])
    elif age >= 65:
        household_size_arr[i] = np.random.choice([1,2,3], p=[0.4,0.4,0.2])
    else:
        household_size_arr[i] = np.random.choice([2,3,4,5,6], p=[0.1,0.3,0.3,0.2,0.1])

# -----------------------------
# Create DataFrame
# -----------------------------
customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "city_id": city_ids,
    "age": ages,
    "gender": genders,
    "marital_status": marital_status,
    "profession": professions_arr,
    "religion": religions_arr,
    "education_level": education_arr,
    "income": income_arr,
    "household_size": household_size_arr,
    "date_joined": date_joined
})

# Preview
display(customers_df.head())

,customer_id,city_id,age,gender,marital_status,profession,religion,education_level,income,household_size,date_joined
0,1,79,47,Female,Married,Working,Christian,High School,54238,5,2017-01-31
1,2,9,45,Female,Widowed,Homemaker,Muslim,High School,40734,2,2022-01-03
2,3,54,52,Male,Married,Working,Hindu,High School,90900,4,2022-05-21
3,4,171,41,Male,Single,Working,Muslim,Bachelor,42176,6,2016-03-16
4,5,81,25,Female,Single,Student,Muslim,Bachelor,3280,2,2017-08-22


In [20]:
file_name = "synthetic_bank_customers.csv"
customers_df.to_csv(file_name, index=False)